# Lecture 11 · Notebook 2 — Quantize a small model and watch its perplexity
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_2_quantize_a_small_model.ipynb)

**Companion to [Chapter 4 of the video lecture](https://nlp-course-cuhksz.github.io/lecture11.html#p4-s53)** (quantization, slides 50–57).
Quantization stores each weight with fewer bits by snapping it to a small grid: $q = \mathrm{round}(w / s)$, $\hat w = s \cdot q$
(slides 52–53). In this notebook you will:

1. snap a few weights to INT4 by hand and see how **one outlier** ruins the grid (slide 53),
2. quantize every linear layer of a real model (SmolLM2-135M) to 8, 4, 3 and 2 bits, with one scale **per tensor**, **per output channel**,
   or **per group of 64 weights**, and measure the perplexity on WikiText-2 — the experiment behind the slide-57 rule of thumb,
3. find the **activation outliers** that LLM.int8() was designed for (slide 54) and see what keeping them in high precision buys.

We use *fake quantization*: weights are rounded to the grid and stored back as floats, so we measure the accuracy effect exactly but not
the speed. Real libraries (bitsandbytes, GPTQ, AWQ, llama.cpp/GGUF) pack the integers and use special kernels.
Runs on **CPU in about 5–10 minutes** (download ~270 MB).

In [ ]:
!pip -q install -U transformers

In [ ]:
import math, torch

import matplotlib.pyplot as plt
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
                     "grid.linewidth": 0.8, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "font.size": 11, "lines.linewidth": 2})

torch.manual_seed(0)

## 1 · Snap eight weights to INT4
Symmetric INT4 has 16 levels, −8 … 7. The scale is the largest absolute weight divided by 7 (slide 53).

In [ ]:
def int4_snap(w):
    s = w.abs().max() / 7
    q = torch.clamp(torch.round(w / s), -8, 7)
    return q, q * s, s

w_good = torch.tensor([0.21, -0.48, 0.05, 0.33, -0.12, 0.40, -0.27, 0.09])
w_bad = w_good.clone(); w_bad[3] = 8.0                       # one outlier
for name, w in [("normal weights", w_good), ("with one outlier", w_bad)]:
    q, wh, s = int4_snap(w)
    err = (w - wh).abs()
    err[w.abs() > 1] = 0                                     # judge the error on the normal weights only
    print(f"{name:17s} scale {s:.3f}  integer codes {q.int().tolist()}")
    print(f"{'':17s} levels used: {len(set(q.int().tolist()))}   mean |error| on normal weights {err.mean():.3f}\n")

With the outlier, the scale stretches to 8.0 / 7 ≈ 1.14, and every normal weight rounds to 0: their information is gone. Two standard
fixes: give **each output channel** its own scale, or go further and give **each small group** of weights its own scale, so an outlier only
spoils its own group (GPTQ and AWQ checkpoints commonly use groups of 128 weights; llama.cpp's GGUF formats use blocks of 32).

## 2 · Quantize a real model
We load SmolLM2-135M and a slice of the WikiText-2 test set, then define a function that rounds every `nn.Linear` weight inside the
Transformer blocks (attention and FFN). The embedding matrix (shared with the output layer) stays in full precision, as in most
weight-only schemes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
SMOL = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(SMOL)
model = AutoModelForCausalLM.from_pretrained(SMOL, dtype=torch.float32).eval()
linears = {n: m for n, m in model.model.layers.named_modules() if isinstance(m, torch.nn.Linear)}
original = {n: m.weight.data.clone() for n, m in linears.items()}
n_q = sum(w.numel() for w in original.values())
print(f"{len(linears)} linear layers, {n_q / 1e6:.1f}M of the model's {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

In [ ]:
import requests, re
# WikiText-2 test set (word-level version from the PyTorch examples repo); "<unk>" marks rare words.
URL = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/test.txt"
raw = requests.get(URL, timeout=60).text
text = " ".join(l.strip() for l in raw.splitlines() if l.strip() and not l.strip().startswith("="))
for a, b in [(" @-@ ", "-"), (" @,@ ", ","), (" @.@ ", "."), (" ,", ","), (" .", "."), (" ;", ";"), (" :", ":"), (" 's", "'s"), ("( ", "("), (" )", ")")]:
    text = text.replace(a, b)
print(len(text.split()), "words; first 300 characters:\n", text[:300])

In [ ]:
enc = tok(text, return_tensors="pt").input_ids[:, :8192]

@torch.no_grad()
def perplexity(window=1024):
    nll, n = 0.0, 0
    for i in range(0, enc.shape[1] - 1, window):
        chunk = enc[:, i:i + window]
        if chunk.shape[1] < 2: break
        nll += model(chunk, labels=chunk).loss.item() * (chunk.shape[1] - 1); n += chunk.shape[1] - 1
    return math.exp(nll / n)

def fake_quant(w, bits, mode, group=64):
    """Symmetric round-to-nearest. mode: 'tensor' (one scale), 'channel' (one per output row), 'group' (one per `group` weights)."""
    qmax = 2 ** (bits - 1) - 1
    if mode == "tensor":
        s = w.abs().max() / qmax
        return torch.clamp(torch.round(w / s), -qmax - 1, qmax) * s
    if mode == "channel":
        s = w.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
        return torch.clamp(torch.round(w / s), -qmax - 1, qmax) * s
    out_f, in_f = w.shape
    g = w.reshape(out_f, in_f // group, group)
    s = g.abs().amax(dim=2, keepdim=True).clamp(min=1e-8) / qmax
    return (torch.clamp(torch.round(g / s), -qmax - 1, qmax) * s).reshape(out_f, in_f)

def quantize_model(bits=None, mode="channel", group=64):
    for n, m in linears.items():
        m.weight.data.copy_(original[n] if bits is None else fake_quant(original[n], bits, mode, group))

def bits_per_weight(bits, mode, group=64):
    return bits + (16 / group if mode == "group" else 0)     # each group stores one FP16 scale

In [ ]:
quantize_model(None)
base = perplexity()
print(f"FP32 baseline perplexity: {base:.2f}\n")
configs = [(8, "tensor"), (8, "channel"), (4, "tensor"), (4, "channel"), (4, "group"),
           (3, "channel"), (3, "group"), (2, "group")]
table = []
for bits, mode in configs:
    quantize_model(bits, mode)
    p = perplexity()
    bpw = bits_per_weight(bits, mode)
    mb = n_q * bpw / 8 / 1e6
    table.append((bits, mode, bpw, p))
    change = f"{p / base - 1:+.1%}" if p < 10 * base else f"{p / base:,.0f}x worse"
    print(f"INT{bits} per-{mode:7s}  {bpw:5.3f} bits/weight  {mb:6.1f} MB for the linear layers   perplexity {p:12,.1f}  ({change})")
quantize_model(None)
print(f"\n(FP16 would be {n_q * 16 / 8 / 1e6:.1f} MB)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for mode, color in [("tensor", ORANGE), ("channel", BLUE), ("group", AQUA)]:
    pts = [(bpw, p) for b, m, bpw, p in table if m == mode]
    ax.plot([x for x, _ in pts], [y for _, y in pts], color=color, marker="o", ms=8, label=f"one scale per {mode}" + (" of 64" if mode == "group" else ""))
ax.axhline(base, color=MUTED, lw=1, ls="--"); ax.text(8.6, base, "FP32", va="center", color=MUTED)
ax.set_yscale("log"); ax.set_xlabel("bits per weight (including scales)"); ax.set_ylabel("perplexity (log scale)")
ax.set_title("SmolLM2-135M, weight-only round-to-nearest quantization", loc="left"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

Read the plot from right to left. INT8 is almost free (slide 57: "almost free, half the memory"). At 4 bits the **granularity of the scale**
matters a lot: one scale per tensor can break the model, per-channel helps, small groups help most. Below 4 bits, plain rounding falls
apart; that is where methods such as GPTQ (error compensation with second-order information) and AWQ (protecting salient channels) come in.
Small models are also *harder* to quantize than large ones, so a 7B model at INT4 loses much less than this 135M model.

## 3 · Activation outliers (LLM.int8())
So far we only quantized weights. To use fast INT8 matrix multiplies you must also quantize the **activations** — the inputs of each
linear layer. Dettmers et al. found that a few hidden dimensions carry values far larger than the rest (slide 54). Let's look at the
inputs to every linear layer of SmolLM2.

In [ ]:
stats = {}
def make_hook(name):
    def hook(mod, inp):
        x = inp[0].detach().abs().reshape(-1, inp[0].shape[-1])
        stats[name] = torch.maximum(stats.get(name, torch.zeros(x.shape[1])), x.amax(0))
    return hook
hooks = [m.register_forward_pre_hook(make_hook(n)) for n, m in linears.items()]
with torch.no_grad():
    model(enc[:, :1024])
for h in hooks: h.remove()

ratio = {n: (s.max() / s.median()).item() for n, s in stats.items()}
worst = max(ratio, key=ratio.get)
print("largest max/median ratio of per-channel |activation|:")
for n in sorted(ratio, key=ratio.get, reverse=True)[:5]:
    print(f"  {n:28s} max {stats[n].max():8.1f}   median {stats[n].median():6.2f}   ratio {ratio[n]:7.1f}")
print("channels above 6.0 (the LLM.int8() threshold) in", worst, ":", int((stats[worst] > 6).sum()), "of", len(stats[worst]))

s = stats[worst]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(range(len(s)), s.numpy(), width=1.0, color=BLUE)
ax.axhline(6, color=ORANGE, lw=1.5); ax.text(len(s), 6, " threshold 6", color=INK, va="bottom", ha="right")
ax.set_xlabel("hidden dimension"); ax.set_ylabel("max |activation|"); ax.set_title(f"inputs to {worst}", loc="left")
plt.tight_layout(); plt.show()

Now quantize those activations to INT8, with **one scale per token** (each row of the input), on top of INT8 per-channel weights
(a "W8A8" model). Then repeat with LLM.int8()'s fix: the few input channels whose magnitude exceeds 6 skip quantization and are
multiplied in full precision; everything else is INT8.

In [ ]:
def act_quant_hook(keep_outliers, bits=8, threshold=6.0):
    qmax = 2 ** (bits - 1) - 1
    def hook(mod, inp):
        x = inp[0]
        if keep_outliers:
            out_cols = (x.abs().amax(dim=tuple(range(x.dim() - 1))) > threshold)   # outlier input channels
            xr = x.masked_fill(out_cols, 0.0)
        else:
            out_cols, xr = None, x
        s = xr.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8) / qmax          # one scale per token
        xq = torch.clamp(torch.round(xr / s), -qmax - 1, qmax) * s
        if keep_outliers:
            xq = torch.where(out_cols, x, xq)                                     # outliers stay in high precision
        return (xq,) + tuple(inp[1:])
    return hook

def run(act_bits, keep_outliers):
    quantize_model(8, "channel")
    hs = [m.register_forward_pre_hook(act_quant_hook(keep_outliers, act_bits)) for m in linears.values()]
    p = perplexity()
    for h in hs: h.remove()
    quantize_model(None)
    return p

print(f"FP32                                   perplexity {base:8.2f}")
for a in [8, 6]:
    print(f"W8A{a}, all activations quantized       perplexity {run(a, False):8.2f}")
    print(f"W8A{a}, outlier channels kept in FP     perplexity {run(a, True):8.2f}")

At 135M parameters the outliers are mild, so plain W8A8 may already work — the LLM.int8() paper showed the breakdown appears around
6.7B parameters, where outlier features become large and systematic. Lowering the activation precision (A6) shows the same effect in
miniature: the fewer the bits, the more it helps to treat the outlier channels separately.

## Exercises
1. Try group sizes 32 and 192 at 3 bits (the group must divide SmolLM2's widths, 576 and 1536). How many bits per weight does each cost, and what does it buy?
2. Implement **asymmetric** quantization with a zero point (slide 52: $r \approx S(q - Z)$). Does it help at 3 bits?
3. Swap in a bigger model, e.g. `Qwen/Qwen2.5-0.5B` (~1 GB). Is INT4 per-channel more or less harmful than for SmolLM2-135M?
4. On a GPU, load a model in real 4-bit with `bitsandbytes` (`BitsAndBytesConfig(load_in_4bit=True)`) and compare its memory with
   `model.get_memory_footprint()`.

**Next:** [Notebook 3 — LoRA parameter counts and training memory](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_3_lora_and_training_memory.ipynb) · **Back to the [video lecture](https://nlp-course-cuhksz.github.io/lecture11.html)**